In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score
import torch

# ==========================================
path = '../../dataset/preprocessed/hotel_bookings_dummy.csv'
df = pd.read_csv(path)

X = df.drop(['is_canceled'], axis=1)
y = df['is_canceled']
X = pd.get_dummies(X, drop_first=True, dtype=float)

# 개별 모델들과 똑같은 기준으로 테스트 데이터를 분리합니다.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    train_size=0.7, 
    random_state=1004
)

print(f"데이터 로드 완료! Test 데이터 크기: {X_test.shape}")

target_path = '../../visualization/model_results'

rf_model = joblib.load(f'{target_path}/best_rf.pkl')
xgb_model = joblib.load(f'{target_path}/best_xgb.pkl')
mlp_model = joblib.load(f'{target_path}/best_mlp.pkl') 

print("모든 진짜 커스텀 모델(RF, XGB, MLP) 로드 성공!")
print("\n각 모델의 실전 데이터 예측 확률 추출 중...")

# (1) 머신러닝 계열 모델 확률 추출
rf_prob = rf_model.predict_proba(X_test)[:, 1]
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

# (2) ⚡ [커널 충돌 차단] MLP 모델 디바이스 'cpu' 강제 타겟팅
# 기존의 cuda 체크 로직 대신 'cpu'를 강제 지정하여 이종 디바이스 충돌을 원천 차단합니다.
target_device = torch.device('cpu')
print(f"연산에 사용할 최종 디바이스 세팅: {target_device}")

# 모델 내부의 모든 레이어 가중치를 깨끗하게 cpu 무대로 밀어 넣습니다.
mlp_model.to(target_device)

# 데이터 텐서도 완벽하게 cpu 디바이스 영역으로 안전하게 생성합니다.
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32).to(target_device)

# 모델을 평가 모드(eval)로 변경
mlp_model.eval()

with torch.no_grad(): # 그라디언트 계산 비활성화로 로컬 RAM 사용량 최적화
    # 모델에 데이터를 통과시킨 후, 시그모이드 함수를 거쳐 0~1 사이의 취소 확률을 생성합니다.
    mlp_output = mlp_model(X_test_tensor)
    
 
    mlp_prob = torch.sigmoid(mlp_output).numpy().flatten()

print("모든 모델의 확률 추출 완료!")

# 🌟 [민규님의 황금 가중치 결합] 에이스 모델인 XGBoost에 50%, RF에 30%, MLP에 20% 부여
voting_prob = (xgb_prob * 0.5) + (rf_prob * 0.3) + (mlp_prob * 0.2)
voting_preds = (voting_prob >= 0.5).astype(int)


print("\n🔍 각 개별 모델 및 앙상블의 최종 실전 성적 계산 중...")

rf_preds = (rf_prob >= 0.5).astype(int)
xgb_preds = (xgb_prob >= 0.5).astype(int)
mlp_preds = (mlp_prob >= 0.5).astype(int)

metrics_summary = []


for name, preds in zip(['Random Forest', 'XGBoost', 'MLP (Deep Learning)'], [rf_preds, xgb_preds, mlp_preds]):
    metrics_summary.append({
        'Model_Name': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1_Score': f1_score(y_test, preds)
    })

# 최종 가중치 보팅 앙상블 점수 추가
metrics_summary.append({
    'Model_Name': ' Weighted Voting Ensemble',
    'Accuracy': accuracy_score(y_test, voting_preds),
    'Recall': recall_score(y_test, voting_preds),
    'F1_Score': f1_score(y_test, voting_preds)
})


df_res = pd.DataFrame(metrics_summary)

display(df_res.round(4))  # 노트북 화면에 이쁜 표로 출력
print("="*80)

df_only_ensemble = df_res[df_res['Model_Name'] == 'Weighted Voting Ensemble'].reset_index(drop=True)

output_dir = '../../visualization/model_results'
os.makedirs(output_dir, exist_ok=True)

# 앙상블 결과 1줄만 담아서 CSV 파일로 깨끗하게 저장합니다.
df_only_ensemble.to_csv(f'{output_dir}/total_ensemble_result.csv', index=False)

#print(f"\n 저장 완료!  '{output_dir}/total_ensemble_result.csv'로 저장되었습니다!")

데이터 로드 완료! Test 데이터 크기: (35671, 51)
모든 진짜 커스텀 모델(RF, XGB, MLP) 로드 성공!

각 모델의 실전 데이터 예측 확률 추출 중...


/opt/anaconda3/lib/python3.13/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


연산에 사용할 최종 디바이스 세팅: cpu
모든 모델의 확률 추출 완료!

🔍 각 개별 모델 및 앙상블의 최종 실전 성적 계산 중...


,Model_Name,Accuracy,Recall,F1_Score
0,Random Forest,0.8771,0.8008,0.8292
1,XGBoost,0.8763,0.8022,0.8284
2,MLP (Deep Learning),0.3723,1.0000,0.5426
3,⭐ Weighted Voting Ensemble,0.8786,0.8031,0.8312
